In [31]:
import joblib
import numpy as np
import os

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter

In [32]:
save_dir = "data/processed"

raw_data    = np.load(os.path.join(save_dir, "raw_data.npy"))
temperature = np.load(os.path.join(save_dir, "temperature.npy"))
train_idx   = np.load(os.path.join(save_dir, "train_idx.npy"))
val_idx     = np.load(os.path.join(save_dir, "val_idx.npy"))
test_idx    = np.load(os.path.join(save_dir, "test_idx.npy"))
scaler      = joblib.load(os.path.join(save_dir, "scaler.pkl"))

num_train_samples = len(train_idx)
num_val_samples   = len(val_idx)
num_test_samples  = len(test_idx)

temp_mean = scaler.mean_[1]
temp_std  = scaler.scale_[1]

print(f"raw_data:    {raw_data.shape}")
print(f"temperature: {temperature.shape}")
print(f"train/val/test: {num_train_samples} / {num_val_samples} / {num_test_samples}")

temperature_normalized = (temperature - temp_mean) / temp_std

raw_data:    (420451, 14)
temperature: (420451,)
train/val/test: 210225 / 105112 / 105114


In [38]:
print("train split mean:", temperature_normalized[:num_train_samples].mean())  # ~0
print("val split mean:  ", temperature_normalized[num_train_samples:num_train_samples+num_val_samples].mean())  # slightly off
print("test split mean: ", temperature_normalized[num_train_samples+num_val_samples:].mean())

train split mean: -4.15324016507081e-15
val split mean:   0.12680263054077132
test split mean:  0.1571616278369348


In [39]:
sampling_rate   = 6
sequence_length = 120
delay           = sampling_rate * (sequence_length + 24 - 1)
batch_size      = 256

class TimeseriesDataset(Dataset):
    def __init__(self, data, targets, sequence_length, sampling_rate, 
                 start_index, end_index, shuffle=False):
        self.data            = data
        self.targets         = targets
        self.sequence_length = sequence_length
        self.sampling_rate   = sampling_rate

        # Valid starting indices: each sequence of length sequence_length
        # sampled every sampling_rate steps needs:
        # (sequence_length - 1) * sampling_rate + 1 rows ahead
        self.indices = np.arange(start_index, end_index)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        start = self.indices[idx]
        # Sample every `sampling_rate` steps for `sequence_length` steps
        steps = np.arange(start, start + self.sequence_length * self.sampling_rate, 
                          self.sampling_rate)
        x = self.data[steps]
        # Target is `delay` steps ahead of the sequence start
        y = self.targets[start + delay]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)


In [40]:
train_dataset = TimeseriesDataset(
    data=raw_data, targets=temperature,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=0, end_index=num_train_samples,
)

val_dataset = TimeseriesDataset(
    data=raw_data, targets=temperature,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=num_train_samples, end_index=num_train_samples + num_val_samples,
)

test_dataset = TimeseriesDataset(
    data=raw_data, targets=temperature,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=num_train_samples + num_val_samples, end_index=len(raw_data) - delay,
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

# Quick sanity check
for inputs, targets in train_loader:
    print("Input shape:", inputs.shape)   # (batch_size, sequence_length, num_features)
    print("Target shape:", targets.shape) # (batch_size,)
    break

Input shape: torch.Size([256, 120, 14])
Target shape: torch.Size([256])


In [41]:
scaler.mean_

array([ 988.74929466,    8.82590329,  282.9050718 ,    4.31331863,
         75.87275476,   13.14569946,    9.19414209,    3.95148184,
          5.81050741,    9.30208943, 1218.45204015,    2.14977462,
          3.56048029,  176.4405232 ])

In [42]:
inputs, targets = next(iter(train_loader))
print("targets range:", targets.min().item(), "to", targets.max().item())
print("targets mean: ", targets.mean().item())
print("targets std:  ", targets.std().item())

targets range: -15.130000114440918 to 31.110000610351562
targets mean:  8.709061622619629
targets std:   9.26315689086914


In [43]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss = 0.0
    total_mae  = 0.0
    n          = 0
    with torch.set_grad_enabled(training):
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            preds = model(inputs)
            loss  = criterion(preds, targets)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * inputs.size(0)
            total_mae  += torch.sum(torch.abs(preds - targets)).item()
            n          += inputs.size(0)
   # mae_celsius = (total_mae / n) * temp_std
    return total_loss / n, total_mae / n   # MAE already in °C


def get_predictions(model, dataset):
    model.eval()
    all_preds     = []
    all_targets   = []
    loader = DataLoader(dataset, batch_size=256, shuffle=False)
    with torch.no_grad():
        for inputs, targets in loader:
            preds = model(inputs.to(device)).cpu().numpy()
            all_preds.append(preds * temp_std + temp_mean)  # un-normalize predictions
            all_targets.append(targets.numpy())              # targets already in °C
    return np.concatenate(all_preds), np.concatenate(all_targets)

In [44]:
class ModelCheckpoint:
    """Saves the best model based on a monitored metric."""
    def __init__(self, filepath, monitor="val_mae", mode="min", verbose=True):
        self.filepath = filepath
        self.monitor  = monitor
        self.verbose  = verbose
        self.best     = float("inf") if mode == "min" else float("-inf")
        self.mode     = mode

    def step(self, metrics, model=None):
        value    = metrics[self.monitor]
        improved = value < self.best if self.mode == "min" else value > self.best
        if improved:
            self.best = value
            torch.save(model.state_dict(), self.filepath)
            if self.verbose:
                print(f"  ✓ Best model saved ({self.monitor}: {value:.2f}°C)")
        return improved


class EarlyStopping:
    """Stops training when a monitored metric stops improving."""
    def __init__(self, monitor="val_mae", patience=5, min_delta=1e-4, mode="min"):
        self.monitor     = monitor
        self.patience    = patience
        self.min_delta   = min_delta
        self.mode        = mode
        self.best        = float("inf") if mode == "min" else float("-inf")
        self.counter     = 0
        self.should_stop = False

    def step(self, metrics, model=None):
        value    = metrics[self.monitor]
        improved = (value < self.best - self.min_delta if self.mode == "min"
                    else value > self.best + self.min_delta)
        if improved:
            self.best    = value
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
                print(f"  Early stopping triggered (no improvement for {self.patience} epochs)")
        return improved


class ReduceLROnPlateau:
    """Wraps PyTorch scheduler with the same callback interface."""
    def __init__(self, optimizer, monitor="val_mae", patience=3, factor=0.5, verbose=True):
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, patience=patience, factor=factor, verbose=verbose
        )
        self.monitor = monitor

    def step(self, metrics, model=None):
        self.scheduler.step(metrics[self.monitor])

In [45]:
# --- Model ---
class DenseModel(nn.Module):
    def __init__(self, sequence_length, num_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(sequence_length * num_features, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.network(x).squeeze(-1)

In [ ]:
# --- Setup ---
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = DenseModel(sequence_length, raw_data.shape[-1]).to(device)
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.MSELoss()
writer    = SummaryWriter(log_dir="runs/jena_dense")

callbacks = [
    ModelCheckpoint("jena_dense_best.pt", monitor="val_mae"),
]

# --- Training loop ---
epochs = 10
for epoch in range(1, epochs + 1):
    train_loss, train_mae = run_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_mae   = run_epoch(model, val_loader,   criterion)

    metrics = {"train_loss": train_loss, "train_mae": train_mae,
               "val_loss":   val_loss,   "val_mae":   val_mae}

    writer.add_scalars("Loss", {"train": train_loss, "val": val_loss}, epoch)
    writer.add_scalars("MAE",  {"train": train_mae,  "val": val_mae},  epoch)

    print(f"Epoch {epoch:02d} — "
          f"train loss: {train_loss:.4f}, train MAE: {train_mae:.2f}°C | "
          f"val loss: {val_loss:.4f}, val MAE: {val_mae:.2f}°C")

    for cb in callbacks:
        cb.step(metrics, model) if isinstance(cb, ModelCheckpoint) else cb.step(metrics)

writer.close()

Epoch 01 — train loss: 15.5332, train MAE: 2.97°C | val loss: 10.3304, val MAE: 2.52°C
  ✓ Best model saved (val_mae: 2.52°C)
Epoch 02 — train loss: 8.5239, train MAE: 2.30°C | val loss: 10.5913, val MAE: 2.57°C
Epoch 03 — train loss: 7.7995, train MAE: 2.20°C | val loss: 10.9547, val MAE: 2.63°C
Epoch 04 — train loss: 7.4076, train MAE: 2.14°C | val loss: 10.7604, val MAE: 2.60°C
Epoch 05 — train loss: 7.1296, train MAE: 2.10°C | val loss: 10.9745, val MAE: 2.63°C
Epoch 06 — train loss: 6.9193, train MAE: 2.07°C | val loss: 11.2765, val MAE: 2.65°C
Epoch 07 — train loss: 6.7494, train MAE: 2.05°C | val loss: 11.2949, val MAE: 2.66°C
Epoch 08 — train loss: 6.5671, train MAE: 2.02°C | val loss: 11.3643, val MAE: 2.67°C
Epoch 09 — train loss: 6.4498, train MAE: 2.00°C | val loss: 11.8149, val MAE: 2.73°C
Epoch 10 — train loss: 6.3271, train MAE: 1.98°C | val loss: 11.6543, val MAE: 2.70°C


In [49]:
# --- Reload best and evaluate ---
model.load_state_dict(torch.load("jena_dense_best.pt", map_location=device))
_, test_mae = run_epoch(model, test_loader, criterion)
print(f"\nTest MAE: {test_mae:.2f}°C")


Test MAE: 2.63°C
